# NB5a — Statistical-only baseline (Vstat in R^5)

The first real number of the evaluation phase. This notebook asks a single question: **how far do
the five statistical features get on their own?** It trains three classical classifiers — logistic
regression, random forest, and gradient boosting — on Vstat alone and reports the full metric suite
on the held-out test split.

**Input:** `vstat_scaled.parquet` (five z-scored features per article, scaler fit on train only).
**Output:** `nb5a_results.parquet` — metrics per classifier, for the comparison table in the thesis.

## Why this notebook exists, and why it runs first

Two reasons, both deliberate.

First, it is a **cleanliness probe**. A prior experiment on an earlier version of this corpus scored
~99.9%, which meant either the data leaked or the models were already sufficient. After the newline
leak was found and fixed (NB2j/NB3), the cheap-signal control baseline fell to a healthy 68.8%. If
five hand-built statistical features now score in a sane range rather than near-perfect, that is
further evidence the corpus is clean. A statistical-only score near 99% would be a red flag to
investigate before going further.

Second, it is the **lower rung of the comparison ladder**. The thesis argues that fusing statistical
features with a contextual embedding beats either alone. To show the fusion adds value, I need the
statistical-only number first. This is that number.

## Why classical classifiers, not a deep network

Five features is a tiny input. A deep network on five inputs would overfit noise and add nothing over
tree ensembles, which are the standard, strong choice for low-dimensional tabular data. Logistic
regression gives a linear reference; random forest and gradient boosting capture non-linear feature
interactions. Together they bound what Vstat alone can do.

## Setup

In [1]:
import pandas as pd, numpy as np, os, glob, json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

SEED = 42
OUT_DIR = '/kaggle/working'

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    print('AVAILABLE /kaggle/input parquet files:')
    for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True): print('   ', p)
    raise FileNotFoundError(preferred)

VSTAT_PATH = find_parquet('/kaggle/input/notebooks/bahaaqassem/nb4-extract-vstat/vstat_scaled.parquet',
                          'vstat_scaled', 'vstat')
vs = pd.read_parquet(VSTAT_PATH)

FEATURES = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']
print('vstat:', vs.shape)
print('splits:', vs['split'].value_counts().to_dict())
print('features:', FEATURES)
print('any NaN left?', int(vs[FEATURES].isna().sum().sum()), '(should be 0 — imputed in NB4)')

vstat: (7101, 9)
splits: {'train': 5363, 'test': 1093, 'val': 645}
features: ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']
any NaN left? 0 (should be 0 — imputed in NB4)


## Split into train / test matrices

I train on the train split and report on test. The split is pair-aware by construction (assigned in
NB3), so no fact card straddles the boundary and the test score is leak-free. The validation split is
left untouched here — these classifiers have little to tune, so I don't need it, and holding it out
keeps it clean for any later model-selection use.

In [2]:
tr = vs[vs['split'] == 'train']
te = vs[vs['split'] == 'test']

Xtr, ytr = tr[FEATURES].to_numpy(), tr['label'].to_numpy()
Xte, yte = te[FEATURES].to_numpy(), te['label'].to_numpy()

print(f'train: {Xtr.shape} | {100*ytr.mean():.1f}% ai')
print(f'test : {Xte.shape} | {100*yte.mean():.1f}% ai')

train: (5363, 5) | 50.7% ai
test : (1093, 5) | 50.7% ai


## Train the three classifiers and score them

All use balanced class weighting to respect the 50.7% AI skew without discarding data. Metrics are
the full suite the thesis reports: accuracy, precision, recall, macro-F1, and AUC-ROC. I keep
macro-F1 as the headline single number because it weights both classes equally.

In [3]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced',
                                             random_state=SEED),
    'RandomForest':       RandomForestClassifier(n_estimators=400, class_weight='balanced',
                                                 random_state=SEED, n_jobs=-1),
    'GradientBoosting':   GradientBoostingClassifier(n_estimators=300, max_depth=3,
                                                     random_state=SEED),
}

def evaluate(name, model):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    return {
        'model': name,
        'accuracy':  accuracy_score(yte, pred),
        'precision': precision_score(yte, pred),
        'recall':    recall_score(yte, pred),
        'macro_f1':  f1_score(yte, pred, average='macro'),
        'auc_roc':   roc_auc_score(yte, proba),
    }, model, pred

results, fitted, preds = [], {}, {}
for name, m in models.items():
    r, fit, pred = evaluate(name, m)
    results.append(r); fitted[name] = fit; preds[name] = pred
    print(f"{name:<20} acc={r['accuracy']:.3f}  macroF1={r['macro_f1']:.3f}  auc={r['auc_roc']:.3f}")

res = pd.DataFrame(results)

LogisticRegression   acc=0.809  macroF1=0.809  auc=0.870
RandomForest         acc=0.834  macroF1=0.834  auc=0.905
GradientBoosting     acc=0.847  macroF1=0.847  auc=0.904


## Results table

In [4]:
show = res.copy()
for c in ['accuracy', 'precision', 'recall', 'macro_f1', 'auc_roc']:
    show[c] = (100 * show[c]).round(1)
print(show.to_string(index=False))

best = res.loc[res['macro_f1'].idxmax()]
print(f"\nbest statistical-only model: {best['model']} "
      f"(macro-F1 {100*best['macro_f1']:.1f}%, AUC {100*best['auc_roc']:.1f}%)")
print('\ncontrol reference (cheap-signal punctuation baseline from NB3): 68.8%')
print('chance: 50.0%')

             model  accuracy  precision  recall  macro_f1  auc_roc
LogisticRegression      80.9       81.3    80.9      80.9     87.0
      RandomForest      83.4       83.0    84.7      83.4     90.5
  GradientBoosting      84.7       84.9    85.0      84.7     90.4

best statistical-only model: GradientBoosting (macro-F1 84.7%, AUC 90.4%)

control reference (cheap-signal punctuation baseline from NB3): 68.8%
chance: 50.0%


## Confusion matrix and per-class report for the best model

Where do the statistical features fail? A skewed confusion matrix (e.g. AI caught but human
over-flagged, or vice versa) tells the thesis which direction Vstat alone is weak in, and motivates
what the neural half should add.

In [5]:
bname = best['model']
bpred = preds[bname]
cm = confusion_matrix(yte, bpred)
print(f'confusion matrix — {bname} (rows = true, cols = pred; 0=human, 1=ai):')
print(f'            pred_human  pred_ai')
print(f'true_human   {cm[0,0]:8d}  {cm[0,1]:7d}')
print(f'true_ai      {cm[1,0]:8d}  {cm[1,1]:7d}')
print()
print(classification_report(yte, bpred, target_names=['human', 'ai'], digits=3))

confusion matrix — GradientBoosting (rows = true, cols = pred; 0=human, 1=ai):
            pred_human  pred_ai
true_human        455       84
true_ai            83      471

              precision    recall  f1-score   support

       human      0.846     0.844     0.845       539
          ai      0.849     0.850     0.849       554

    accuracy                          0.847      1093
   macro avg      0.847     0.847     0.847      1093
weighted avg      0.847     0.847     0.847      1093



## Feature importance — which of the five carries the signal?

For the tree models I read off importances; for logistic regression, the absolute standardized
coefficients. This is one of the payoffs of the statistical track: unlike the neural embedding, these
features are interpretable, so the thesis can say *which* stylometric property separates the classes,
not just that something does.

In [6]:
imp = pd.DataFrame({'feature': FEATURES})

if 'RandomForest' in fitted:
    imp['random_forest'] = fitted['RandomForest'].feature_importances_
if 'GradientBoosting' in fitted:
    imp['gradient_boosting'] = fitted['GradientBoosting'].feature_importances_
lr = fitted['LogisticRegression']
imp['logreg_abs_coef'] = np.abs(lr.coef_[0])

# normalize each column to sum to 1 for comparability
for c in imp.columns[1:]:
    imp[c] = (imp[c] / imp[c].sum()).round(3)

imp['mean_rank'] = imp[imp.columns[1:]].rank(ascending=False).mean(axis=1)
imp = imp.sort_values('mean_rank')
print('feature importance (normalized; higher = more discriminative):\n')
print(imp.to_string(index=False))
print('\nmost discriminative:', imp.iloc[0]['feature'])

feature importance (normalized; higher = more discriminative):

            feature  random_forest  gradient_boosting  logreg_abs_coef  mean_rank
         burstiness          0.424              0.641            0.520   1.000000
                ttr          0.181              0.155            0.288   2.000000
discourse_coherence          0.141              0.054            0.092   3.666667
       targeted_ppl          0.141              0.099            0.008   3.833333
     entity_density          0.113              0.052            0.092   4.500000

most discriminative: burstiness


## Per-generator breakdown — which models are hardest to catch?

A single accuracy number hides variation across generators. Splitting the AI test articles by their
source model shows whether Vstat catches, say, DeepSeek easily but misses Gemini. This previews the
Leave-One-Generator-Out story and flags any generator whose text is statistically closest to human.

In [7]:
te2 = te.copy()
te2['pred'] = preds[bname]
ai_te = te2[te2['label'] == 1]

print(f'AI-class recall by generator ({bname}):\n')
print(f"{'generator':<12}{'n':>5}{'caught':>8}{'recall':>9}")
for g, sub in ai_te.groupby('generator'):
    rec = (sub['pred'] == 1).mean()
    print(f'{g:<12}{len(sub):>5}{int((sub["pred"]==1).sum()):>8}{100*rec:>8.0f}%')

human_te = te2[te2['label'] == 0]
print(f"\nhuman-class recall (correctly kept as human): "
      f"{100*(human_te['pred']==0).mean():.0f}%")

AI-class recall by generator (GradientBoosting):

generator       n  caught   recall
deepseek      135     104      77%
gemini         63      62      98%
gpt            68      53      78%
opus           63      49      78%
qwen           88      85      97%
sonnet        137     118      86%

human-class recall (correctly kept as human): 84%


## Save

In [8]:
OUT = f'{OUT_DIR}/nb5a_results.parquet'
res.to_parquet(OUT, index=False)
imp.to_parquet(f'{OUT_DIR}/nb5a_feature_importance.parquet', index=False)

print('saved:', OUT)
print('\n=== NB5a summary ===')
print(f'statistical-only, best = {bname}')
print(f'  accuracy {100*best["accuracy"]:.1f}%  macro-F1 {100*best["macro_f1"]:.1f}%  '
      f'AUC {100*best["auc_roc"]:.1f}%')
print(f'  vs cheap-signal control 68.8%, vs chance 50.0%')
print('\nThis is the statistical-only rung. NB5b/c/d give the neural-only rungs;')
print('the hybrid must beat all of them for the fusion thesis to hold.')

saved: /kaggle/working/nb5a_results.parquet

=== NB5a summary ===
statistical-only, best = GradientBoosting
  accuracy 84.7%  macro-F1 84.7%  AUC 90.4%
  vs cheap-signal control 68.8%, vs chance 50.0%

This is the statistical-only rung. NB5b/c/d give the neural-only rungs;
the hybrid must beat all of them for the fusion thesis to hold.


## Notes

**Reading the result:**

- **A sane score (roughly 75–88%) is the good outcome.** It says the five features carry real but
  incomplete signal — exactly the premise of the hybrid model. Near-99% would signal residual
  leakage; near-chance would mean the features are worthless (the raw-mean directions in NB4 already
  argue against that).
- **This is the floor the hybrid must clear**, alongside the neural-only baselines in NB5b/c/d.

**For the thesis:**

- Report all three classifiers, not just the best, so the reader sees the linear-vs-nonlinear gap.
- The feature-importance table is the interpretability payoff of the statistical track — name the
  top feature(s) explicitly. From the NB4 raw means, targeted perplexity and burstiness are the
  strongest separators; TTR is expected to rank low and may even mislead, since modern LLMs are
  lexically richer than humans (its class direction was reversed in NB4).
- The per-generator recall table previews Leave-One-Generator-Out and identifies the
  hardest-to-detect generator on statistical grounds alone.

**Method note:** all classifiers use `class_weight='balanced'`; no subsampling. Metrics are on the
pair-aware test split, so they are directly comparable to every other notebook in the evaluation
phase.